# 공모전 추천 — 모델 개발 노트북

**서버와 동일한 `ContestRecommender` 클래스를 그대로 import 해서 사용**한다.
노트북에서 실험한 코드 = PyCharm에서 실행되는 서버 코드 → 한 곳에서 고치면 양쪽 모두 반영.

## 분담
| 위치 | 역할 |
|------|------|
| `recommend_server/recommender.py` | **추천 엔진 단일 진실** (이 노트북도 여기서 import) |
| `recommend_server/db.py` | DB 엔진 빌더 |
| `recommend_server/main.py` | FastAPI 서버 진입점 (PyCharm에서 실행) |
| `d_ContestRecommend.ipynb` (이 파일) | 위 클래스를 호출해 결과 검증 |
| `contest_recommend_init.sql` | PostgreSQL 테이블 + 샘플 데이터 |

## 사전 준비
1. PostgreSQL 실행
2. SQL 초기화: `psql -U postgres -d <DB명> -f contest_recommend_init.sql`
3. `.env`:
   ```
   DB_USER=postgres
   DB_PASSWORD=...
   DB_HOST=localhost
   DB_PORT=5432
   DB_NAME=test
   ```

## 0. 라이브러리

In [1]:
# %pip install -q pandas sqlalchemy psycopg2-binary scikit-learn kiwipiepy python-dotenv

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

## 1. PostgreSQL 데이터 미리보기

DB에 데이터가 잘 들어갔는지 빠르게 확인. 추천 로직은 `ContestRecommender` 안에 있으므로 여기선 검수만 한다.

In [3]:
import sys
from pathlib import Path

# 서버 코드 폴더를 import 경로에 추가 — recommender / db 를 직접 가져다 쓴다
SERVER_DIR = Path("recommend_server").resolve()
if str(SERVER_DIR) not in sys.path:
    sys.path.insert(0, str(SERVER_DIR))

from db import build_engine  # noqa: E402
import pandas as pd

engine = build_engine()
df = pd.read_sql("SELECT * FROM tbl_recommend_contest ORDER BY id", engine)
print(f"공모전 {len(df)}건 로드")
df.head()

공모전 100건 로드


,id,title,organizer,category,description,entry_count,view_count,created_at
0,1,청년 단편 영상 공모전,문화체육관광부,영상,청년의 시선으로 바라본 일상을 5분 이내 단편 영상으로 표현해 주세요.,120,3500,2026-04-28 10:27:21.614088
1,2,환경 다큐멘터리 공모전,환경재단,영상,기후위기와 생태계를 주제로 한 단편 다큐멘터리 영상을 모집합니다.,85,2800,2026-04-28 10:27:21.614088
2,3,여행 브이로그 영상 공모전,한국관광공사,영상,국내 여행지의 매력을 담은 브이로그 형식의 짧은 영상을 공모합니다.,210,6200,2026-04-28 10:27:21.614088
3,4,학생 애니메이션 단편 공모전,문화예술위원회,영상,중고생을 대상으로 한 2D/3D 단편 애니메이션 영상 공모전입니다.,65,1900,2026-04-28 10:27:21.614088
4,5,브랜드 광고 영상 공모전,마케팅협회,영상,30초 이내 브랜드 광고 영상을 통해 창의력을 발휘해 보세요.,140,4200,2026-04-28 10:27:21.614088


## 2. 추천 엔진 import + 학습

서버가 사용하는 `ContestRecommender` 클래스를 그대로 가져와서 노트북 안에서 fit한다.
→ 서버 시작 시점의 동작과 **완벽히 동일**한 결과를 얻는다.

In [4]:
from recommender import ContestRecommender  # noqa: E402

recommender = ContestRecommender()
recommender.fit()

print(f"인덱스 로드됨: {recommender.is_loaded}")
print(f"문서 수      : {len(recommender.df)}")
print(f"TF-IDF shape : {recommender.tfidf_matrix.shape}")
print(f"어휘 크기    : {len(recommender.vectorizer.vocabulary_)}")

인덱스 로드됨: True
문서 수      : 100
TF-IDF shape : (100, 396)
어휘 크기    : 396


## 3. 토크나이저 결과 확인

`recommender._tokenize()` 가 한국어를 어떻게 자르는지 단일 케이스로 확인.

In [5]:
sample = df.iloc[0]['title'] + ' ' + df.iloc[0]['description']
print("원문:", sample)
print("토큰:", recommender._tokenize(sample))

원문: 청년 단편 영상 공모전 청년의 시선으로 바라본 일상을 5분 이내 단편 영상으로 표현해 주세요.
토큰: 청년 단편 영상 공모전 청년 시선 바라보다 일상 분 이내 단편 영상 표현 하 주다


## 4. 추천 결과 검증

여러 카테고리 ID로 결과를 보면서 품질을 눈으로 확인.
`recommender.recommend()` = FastAPI `/recommend/{id}` 가 호출하는 메서드 그대로.

In [6]:
def show(contest_id, top_k=5):
    result = recommender.recommend(contest_id, top_k)
    if result is None:
        print(f"ID={contest_id} 없음")
        return
    base = result['base']
    print(f"\n[기준] [{base['id']}] {base['title']} ({base['category']})")
    print(f"  가중평균 예상 참여자: {result['predictedPopularity']:.1f}")
    print("-" * 70)
    for it in result['items']:
        print(f"  [{it['id']:3d}] sim={it['similarity']:.3f}  {it['title']:30s}  ({it['category']})")

for test_id in [1, 10, 17, 21, 6]:
    show(test_id, top_k=5)


[기준] [1] 청년 단편 영상 공모전 (영상)
  가중평균 예상 참여자: 92.7
----------------------------------------------------------------------
  [ 96] sim=0.268  청년 자작 시 공모전                     (문학)
  [  4] sim=0.254  학생 애니메이션 단편 공모전                 (영상)
  [ 12] sim=0.238  청소년 단편 영화제                      (영상)
  [ 85] sim=0.215  단편 소설 문학 공모전                    (문학)
  [ 14] sim=0.180  소셜미디어 숏폼 공모전                    (영상)

[기준] [10] 환경 캠페인 영상 공모전 (영상)
  가중평균 예상 참여자: 72.0
----------------------------------------------------------------------
  [ 23] sim=0.384  환경 포스터 디자인 공모전                  (디자인)
  [ 16] sim=0.311  안전 캠페인 영상전                      (영상)
  [  2] sim=0.258  환경 다큐멘터리 공모전                    (영상)
  [100] sim=0.234  환경 동화 공모전                       (문학)
  [ 25] sim=0.176  친환경 제품 디자인 공모전                  (디자인)

[기준] [17] 농촌 소개 영상 공모전 (영상)
  가중평균 예상 참여자: 80.4
----------------------------------------------------------------------
  [ 50] sim=0.319  농촌 사진 공모전                       (사진)
  [ 13] sim=0.274  기업 

## 5. 인덱스 재구축 (DB 변경 후)

DB에 데이터를 추가/수정한 뒤 다시 학습하려면 `fit()` 한 번 더 호출.

In [7]:
recommender.fit()
print(f"재학습 완료: {len(recommender.df)}건")

재학습 완료: 100건


## 6. 서버 실행 (별도 프로세스)

이 노트북은 검증까지만. 실제 서비스는 PyCharm에서 `recommend_server/main.py` 를 따로 실행:

```bash
cd C:\gb_0900_ljs\ai\workspace\llm\recommend_server
pip install -r requirements.txt
python main.py
```

기동 후:
- 헬스체크: http://localhost:8000/health
- API 문서: http://localhost:8000/docs
- Spring 프론트(:10000)와 연동되어 화면에 추천 결과 표시

→ 서버는 위 노트북에서 import한 **동일한 `ContestRecommender` 클래스**를 사용한다.